In [1]:
# Imports & Configuration

import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# GitHub repository
BASE_URL = "https://raw.githubusercontent.com/semoglou/cart/main"

# CART taxonomy
URL_CART = (
    f"{BASE_URL}/taxonomy/"
    "narrative_taxonomy_ksil_with_paragraphs.csv"
)

# DeepSeek-R1 taxonomy
URL_DEEPSEEK = (
    f"{BASE_URL}/llm_baselines/llm_taxonomies/deepseek_r1/"
    "02_classified_dataframe_R1.csv"
)

In [2]:
# Load Taxonomies

import requests
from io import BytesIO

def load_github_csv(url, name):

    print(f"Loading {name}...")

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    df = pd.read_csv(
        BytesIO(response.content),
        dtype={"paragraph_id": "string"}
    )

    print(f"  Loaded {len(df):,} rows")
    print(f"  Columns: {df.columns.tolist()}")
    print()

    return df


df_cart = load_github_csv(
    URL_CART,
    "CART (K-SIL)"
)

df_deepseek = load_github_csv(
    URL_DEEPSEEK,
    "DeepSeek-R1"
)

Loading CART (K-SIL)...
  Loaded 4,537 rows
  Columns: ['paragraph_id', 'paragraph', 'narrative', 'narratives_label', 'narrative_label_clean']

Loading DeepSeek-R1...
  Loaded 10,686 rows
  Columns: ['paragraph_id', 'sitting_date', 'political_party', 'paragraph', 'narrative', 'narrative_family']



In [3]:
# Statistics

datasets = {
    "CART (K-SIL)": (
        df_cart,
        "narrative_label_clean"
    ),
    "DeepSeek-R1": (
        df_deepseek,
        "narrative_family"
    )
}

stats = []

for name, (df, label_col) in datasets.items():

    required = [
        "paragraph_id",
        "paragraph",
        label_col
    ]

    missing = [
        col for col in required
        if col not in df.columns
    ]

    assert not missing, (
        f"{name}: Missing columns {missing}"
    )

    stats.append({
        "System": name,
        "Total rows": len(df),
        "Unique paragraphs": df["paragraph_id"].nunique(),
        "Narrative families": df[label_col].nunique(),
        "Duplicate IDs": df["paragraph_id"].duplicated().sum(),
        "Missing IDs": df["paragraph_id"].isna().sum(),
        "Missing paragraphs": df["paragraph"].isna().sum(),
        "Missing labels": df[label_col].isna().sum()
    })

stats_df = pd.DataFrame(stats)

display(stats_df)

,System,Total rows,Unique paragraphs,Narrative families,Duplicate IDs,Missing IDs,Missing paragraphs,Missing labels
0,CART (K-SIL),4537,4537,10,0,0,0,0
1,DeepSeek-R1,10686,10685,21,1,0,0,0


In [4]:
# Narrative Family Distributions

for name, (df, label_col) in datasets.items():

    print("=" * 70)
    print(name)
    print("=" * 70)

    family_counts = (
        df[label_col]
        .value_counts(dropna=False)
        .rename_axis("Narrative Family")
        .reset_index(name="Paragraphs")
    )

    family_counts["Percentage (%)"] = (
        family_counts["Paragraphs"] / len(df) * 100
    ).round(2)

    display(family_counts)

    print(
        f"Total families: {df[label_col].nunique()}"
    )
    print()

CART (K-SIL)


,Narrative Family,Paragraphs,Percentage (%)
0,Government cover-up of Tempi accountability,1058,23.32
1,Railway safety remains neglected and unsecured,837,18.45
2,Judicial and procedural obstacles to accountability,627,13.82
3,Systemic governance failures caused preventable disaster,603,13.29
4,Government obstructs crash investigation to evade accountability,382,8.42
5,Negligence caused the preventable Tempi train crash,362,7.98
6,Tempi crash resulted from systemic criminal failure,236,5.20
7,Opposition exploits the tragedy for political gain,170,3.75
8,Legal framework shields ministers from accountability,142,3.13
9,Railway privatization caused systemic safety failures,120,2.64


Total families: 10

DeepSeek-R1


,Narrative Family,Paragraphs,Percentage (%)
0,Systemic Governance Negligence Causation,2678,25.06
1,Investigation Process Deficiencies,2175,20.35
2,Institutional Corruption and Impunity,2097,19.62
3,Political Exploitation Undermining Accountability,670,6.27
4,Independent Investigative Oversight Demands,664,6.21
5,Legal and Institutional Reform Imperatives,610,5.71
6,Uncategorized,409,3.83
7,Judicial Integrity Requirements,217,2.03
8,Institutional Competence Critiques,181,1.69
9,Criminal Negligence Liability Allegations,170,1.59


Total families: 21



In [5]:
# Remove Duplicate Paragraph IDs Safely

def clean_dataset(df, name, label_col):

    df = df.copy()

    # Normalize IDs and remove surrounding whitespace
    df["paragraph_id"] = (
        df["paragraph_id"]
        .astype("string")
        .str.strip()
    )

    # Detect duplicate IDs
    duplicate_mask = df["paragraph_id"].duplicated(
        keep=False
    )

    duplicate_rows = df[duplicate_mask].copy()

    if len(duplicate_rows) > 0:

        print(f"\n{name}: Duplicate IDs detected!")

        display(
            duplicate_rows[
                ["paragraph_id", "paragraph", label_col]
            ].sort_values("paragraph_id")
        )

        print(
            f"Removing {duplicate_rows['paragraph_id'].nunique()} "
            "ambiguous paragraph IDs."
        )

    # Exclude ALL occurrences of duplicate IDs
    df = df[~duplicate_mask].copy()

    # Remove rows with missing required fields
    df = df.dropna(
        subset=[
            "paragraph_id",
            "paragraph",
            label_col
        ]
    )

    # Exclude empty strings
    for col in ["paragraph_id", "paragraph", label_col]:
        df = df[
            df[col].astype(str).str.strip().ne("")
        ]

    # Validation
    assert df["paragraph_id"].is_unique
    assert df["paragraph_id"].notna().all()

    print(
        f"{name}: {len(df):,} valid unique paragraphs"
    )

    return df.reset_index(drop=True)


cart_clean = clean_dataset(
    df_cart,
    "CART",
    "narrative_label_clean"
)

deepseek_clean = clean_dataset(
    df_deepseek,
    "DeepSeek-R1",
    "narrative_family"
)

CART: 4,537 valid unique paragraphs

DeepSeek-R1: Duplicate IDs detected!


,paragraph_id,paragraph,narrative_family
7293,2025-03-05_314_389_p1,"Ευχαριστώ, κύριε Πρόεδρε. Κυρίες και κύριοι συνάδελφοι, είναι η δεύτερη φορά μέσα σε έναν χρόνο ...",Independent Investigative Oversight Demands
7294,2025-03-05_314_389_p1,"Ευχαριστώ, κύριε Πρόεδρε. Κυρίες και κύριοι συνάδελφοι, είναι η δεύτερη φορά μέσα σε έναν χρόνο ...",Institutional Corruption and Impunity


Removing 1 ambiguous paragraph IDs.
DeepSeek-R1: 10,684 valid unique paragraphs


In [6]:
# Intersection Analysis — CART vs DeepSeek-R1

ids_cart = set(cart_clean["paragraph_id"])
ids_deepseek = set(deepseek_clean["paragraph_id"])

# Common paragraphs between CART and DeepSeek-R1
common_ids = ids_cart & ids_deepseek

intersection_stats = pd.DataFrame([
    {
        "Comparison": "CART ∩ DeepSeek-R1",
        "Common paragraphs": len(common_ids)
    }
])

display(intersection_stats)

print("\n" + "=" * 60)
print(
    f"CART–DEEPSEEK INTERSECTION: "
    f"{len(common_ids):,} paragraphs"
)
print("=" * 60)

,Comparison,Common paragraphs
0,CART ∩ DeepSeek-R1,3803



CART–DEEPSEEK INTERSECTION: 3,803 paragraphs


In [7]:
# Merge & Final Validation — CART vs DeepSeek-R1

# Select and rename columns

cart_final = cart_clean[
    ["paragraph_id", "paragraph", "narrative_label_clean"]
].rename(columns={
    "paragraph": "paragraph_cart",
    "narrative_label_clean": "nf_cart"
})

deepseek_final = deepseek_clean[
    ["paragraph_id", "paragraph", "narrative_family"]
].rename(columns={
    "paragraph": "paragraph_deepseek",
    "narrative_family": "nf_deepseek"
})


# Inner join: paragraphs present in both systems

merged = cart_final.merge(
    deepseek_final,
    on="paragraph_id",
    how="inner",
    validate="one_to_one"
)


# Normalize paragraph text for comparison only

def normalize_text(series):
    return series.astype("string").str.strip()


cart_text = normalize_text(merged["paragraph_cart"])
deepseek_text = normalize_text(merged["paragraph_deepseek"])

matching_texts = cart_text.eq(deepseek_text)

mismatches = merged[~matching_texts.fillna(False)]

print(f"Merged paragraphs: {len(merged):,}")
print(f"Matching paragraph texts: {matching_texts.sum():,}")
print(f"Text mismatches: {len(mismatches):,}")


# Stop if the same ID refers to different text

if len(mismatches) > 0:

    display(
        mismatches[
            [
                "paragraph_id",
                "paragraph_cart",
                "paragraph_deepseek"
            ]
        ].head(10)
    )

    raise ValueError(
        "Paragraph text mismatches detected. "
        "Review the data before exporting."
    )


# Build final dataset

final_df = merged[
    [
        "paragraph_id",
        "paragraph_cart",
        "nf_cart",
        "nf_deepseek"
    ]
].rename(columns={
    "paragraph_cart": "paragraph"
})


# Final validation

assert len(final_df) == len(common_ids)
assert final_df["paragraph_id"].is_unique
assert final_df.notna().all().all()

assert (
    final_df["paragraph_id"].nunique()
    == len(final_df)
)

final_df = final_df.sort_values(
    "paragraph_id"
).reset_index(drop=True)


print("\n" + "=" * 60)
print("FINAL DATASET VALIDATION")
print("=" * 60)

print(f"Total paragraphs: {len(final_df):,}")
print(f"Unique IDs: {final_df['paragraph_id'].nunique():,}")
print(f"Duplicate IDs: {final_df['paragraph_id'].duplicated().sum()}")
print(f"Missing values: {final_df.isna().sum().sum()}")

print("\nNarrative families represented in intersection:")

print(
    f"CART: {final_df['nf_cart'].nunique()}"
)

print(
    f"DeepSeek-R1: {final_df['nf_deepseek'].nunique()}"
)

print("\nFinal dataset preview:")

display(final_df.head(10))

Merged paragraphs: 3,803
Matching paragraph texts: 3,803
Text mismatches: 0

FINAL DATASET VALIDATION
Total paragraphs: 3,803
Unique IDs: 3,803
Duplicate IDs: 0
Missing values: 0

Narrative families represented in intersection:
CART: 10
DeepSeek-R1: 21

Final dataset preview:


,paragraph_id,paragraph,nf_cart,nf_deepseek
0,2023-03-07_1_10_p3,"Νομίζω ότι είναι για όλους μας, για τη χώρα μας, για όλες τις Ελληνίδες, για όλους τους Έλληνες,...",Railway safety remains neglected and unsecured,Political Responsibility for Systemic Failures
1,2023-03-07_1_10_p4,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μεγάλες ευθύνες που υπάρχουν που αφορούν την Κυβέρν...,Systemic governance failures caused preventable disaster,Systemic Governance Negligence Causation
2,2023-03-07_1_10_p6,"Η κοινωνία πρέπει να μάθει όλη την αλήθεια. Πρέπει, λοιπόν, να αποδοθούν ευθύνες προς κάθε κατεύ...",Railway safety remains neglected and unsecured,Uncategorized
3,2023-03-07_1_18_p2,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κατ’ αρχάς, θέλω να εκφράσω τη θλίψη, αλλά και την ...",Government obstructs crash investigation to evade accountability,Institutional Corruption and Impunity
4,2023-03-07_1_18_p4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα κάτω από το βάρος μιας εθνικής τραγωδίας. Άραγε, ...",Negligence caused the preventable Tempi train crash,Independent Investigative Oversight Demands
5,2023-03-07_1_18_p7,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τελευταία τέσσερα χρόνια που κυβερνάει στον τομέα το...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
6,2023-03-07_1_18_p8,"Πριν από έξι με επτά μήνες περίπου είχαμε μάθει ότι ακόμα και ο προαστιακός, κύριε Υπουργέ -και ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
7,2023-03-07_1_18_p9,"Νέα παιδιά τα οποία παίρνουν το τρένο, φοιτητές, στρατιώτες, μαθητές παραγωγικών σχολών χάθηκαν ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
8,2023-03-07_1_29_p1,"Κύριε Πρόεδρε, αν και τα λόγια είναι περιττά σε αυτές τις περιπτώσεις γιατί η θλίψη είναι τεράστ...",Judicial and procedural obstacles to accountability,Institutional Corruption and Impunity
9,2023-03-07_1_29_p3,"Ε.». Βεβαίως, αυτή η τραγωδία ήταν προδιαγεγραμμένη από όταν και ο ΣΥΡΙΖΑ αποδέχτηκε αυτές τις μ...",Negligence caused the preventable Tempi train crash,Systemic Governance Negligence Causation


In [8]:
# Remove Uncategorized Labels — CART vs DeepSeek-R1

label_cols = ["nf_cart", "nf_deepseek"]

# Identify rows where either system has "Uncategorized"

uncategorized_mask = (
    final_df[label_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.casefold())
    .eq("uncategorized")
    .any(axis=1)
)

# Report excluded rows

excluded_df = final_df.loc[uncategorized_mask].copy()

print("=" * 60)
print("UNCATEGORIZED FILTERING")
print("=" * 60)

print(f"Original shared paragraphs: {len(final_df):,}")
print(f"Excluded paragraphs: {len(excluded_df):,}")

for col in label_cols:

    count = (
        final_df[col]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("uncategorized")
        .sum()
    )

    print(f"{col}: {count:,} Uncategorized labels")


# Keep only rows with valid labels in both systems

final_df = final_df.loc[~uncategorized_mask].copy()
final_df = final_df.reset_index(drop=True)

print(f"\nFinal categorized paragraphs: {len(final_df):,}")


# Final validation

assert final_df["paragraph_id"].is_unique
assert final_df[label_cols].notna().all().all()

assert not (
    final_df[label_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.casefold())
    .eq("uncategorized")
    .any(axis=1)
    .any()
)

display(final_df.head(10))

UNCATEGORIZED FILTERING
Original shared paragraphs: 3,803
Excluded paragraphs: 147
nf_cart: 0 Uncategorized labels
nf_deepseek: 147 Uncategorized labels

Final categorized paragraphs: 3,656


,paragraph_id,paragraph,nf_cart,nf_deepseek
0,2023-03-07_1_10_p3,"Νομίζω ότι είναι για όλους μας, για τη χώρα μας, για όλες τις Ελληνίδες, για όλους τους Έλληνες,...",Railway safety remains neglected and unsecured,Political Responsibility for Systemic Failures
1,2023-03-07_1_10_p4,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μεγάλες ευθύνες που υπάρχουν που αφορούν την Κυβέρν...,Systemic governance failures caused preventable disaster,Systemic Governance Negligence Causation
2,2023-03-07_1_18_p2,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κατ’ αρχάς, θέλω να εκφράσω τη θλίψη, αλλά και την ...",Government obstructs crash investigation to evade accountability,Institutional Corruption and Impunity
3,2023-03-07_1_18_p4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα κάτω από το βάρος μιας εθνικής τραγωδίας. Άραγε, ...",Negligence caused the preventable Tempi train crash,Independent Investigative Oversight Demands
4,2023-03-07_1_18_p7,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τελευταία τέσσερα χρόνια που κυβερνάει στον τομέα το...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
5,2023-03-07_1_18_p8,"Πριν από έξι με επτά μήνες περίπου είχαμε μάθει ότι ακόμα και ο προαστιακός, κύριε Υπουργέ -και ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
6,2023-03-07_1_18_p9,"Νέα παιδιά τα οποία παίρνουν το τρένο, φοιτητές, στρατιώτες, μαθητές παραγωγικών σχολών χάθηκαν ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
7,2023-03-07_1_29_p1,"Κύριε Πρόεδρε, αν και τα λόγια είναι περιττά σε αυτές τις περιπτώσεις γιατί η θλίψη είναι τεράστ...",Judicial and procedural obstacles to accountability,Institutional Corruption and Impunity
8,2023-03-07_1_29_p3,"Ε.». Βεβαίως, αυτή η τραγωδία ήταν προδιαγεγραμμένη από όταν και ο ΣΥΡΙΖΑ αποδέχτηκε αυτές τις μ...",Negligence caused the preventable Tempi train crash,Systemic Governance Negligence Causation
9,2023-03-07_1_31_p1,"Ευχαριστώ πολύ, κύριε Πρόεδρε. Κυρίες και κύριοι συνάδελφοι, η σημερινή συνεδρίαση διεξάγεται κά...",Judicial and procedural obstacles to accountability,Systemic Governance Negligence Causation


In [9]:
# Export Final Dataset

from google.colab import files

output_filename = "CART_DeepSeek_R1_shared_paragraphs.csv"

final_df.to_csv(
    output_filename,
    index=False,
    encoding="utf-8-sig"
)

print(f"Filename: {output_filename}")
print(f"Paragraphs: {len(final_df):,}")
print(f"Columns: {final_df.columns.tolist()}")

files.download(output_filename)

Filename: CART_DeepSeek_R1_shared_paragraphs.csv
Paragraphs: 3,656
Columns: ['paragraph_id', 'paragraph', 'nf_cart', 'nf_deepseek']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>